# Módulo 3 — EDA: Sistema de Recomendación de Destinos
**Dataset real**: `amanmehra23/travel-recommendation-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, sys
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
OUTPUT_DIR = Path('.'); print("Librerías cargadas.")

## 1. Carga del dataset real

In [ ]:
import kagglehub
try:
    raw_path = Path(kagglehub.dataset_download("amanmehra23/travel-recommendation-dataset"))
except Exception as e:
    sys.exit(f"ERROR: {e}\nConfigura ~/.kaggle/kaggle.json")
csvs = {p.stem.lower(): p for p in raw_path.rglob("*.csv")}
reviews_path = next((p for k,p in csvs.items() if "review" in k), None)
dest_path    = next((p for k,p in csvs.items() if "destination" in k), None)
assert reviews_path and dest_path
df_reviews = pd.read_csv(reviews_path); df_dest = pd.read_csv(dest_path)
print(f"Reviews: {df_reviews.shape}  |  Destinations: {df_dest.shape}")
print("Reviews columnas:", list(df_reviews.columns))
print(df_reviews.head(3))

## 2. Distribución de ratings

In [ ]:
rating_col = next((c for c in df_reviews.columns if 'rating' in c.lower() or 'score' in c.lower()), None)
assert rating_col, f"No rating col. Cols: {list(df_reviews.columns)}"
print(f"Columna rating: {rating_col}")
print(df_reviews[rating_col].describe())

fig, axes = plt.subplots(1,2,figsize=(12,4))
df_reviews[rating_col].hist(bins=20, color='#3182CE', edgecolor='white', ax=axes[0])
axes[0].set_title("Distribución de Ratings"); axes[0].set_xlabel("Rating")
axes[0].axvline(df_reviews[rating_col].mean(), color='red', linestyle='--', label=f"Media: {df_reviews[rating_col].mean():.2f}")
axes[0].legend()
vc=df_reviews[rating_col].round().value_counts().sort_index()
axes[1].bar(vc.index.astype(str), vc.values, color='#48BB78', edgecolor='white')
axes[1].set_title("Frecuencia por rating (redondeado)"); axes[1].set_xlabel("Rating")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig_rating_dist.png",dpi=120,bbox_inches='tight')
plt.show()

## 3. Top 20 destinos más reseñados

In [ ]:
user_col = next(c for c in df_reviews.columns if 'user' in c.lower())
item_col = next(c for c in df_reviews.columns if 'dest' in c.lower() or 'item' in c.lower())
print(f"User: {user_col} | Item: {item_col}")

# Join con nombre del destino
if item_col=="DestinationID" and "Name" in df_dest.columns:
    merged = df_reviews.merge(df_dest[["DestinationID","Name","Type","State"]].drop_duplicates("DestinationID"),
                              on="DestinationID", how="left")
    name_col="Name"
else:
    merged=df_reviews.copy(); name_col=item_col

top20 = merged[name_col].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12,7))
ax.barh(top20.index[::-1], top20.values[::-1], color='#3182CE')
ax.set_xlabel("Número de reseñas"); ax.set_title("Top 20 Destinos más Reseñados")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig_top20_destinos.png",dpi=120,bbox_inches='tight')
plt.show()

## 4. Actividad por usuario y sparsity de la matriz

In [ ]:
user_activity = df_reviews[user_col].value_counts()
print(f"Usuarios únicos: {df_reviews[user_col].nunique()}")
print(f"Items únicos:    {df_reviews[item_col].nunique()}")
n_users_total = df_reviews[user_col].nunique()
n_items_total = df_reviews[item_col].nunique()
sparsity = 1 - len(df_reviews) / (n_users_total * n_items_total)
print(f"Interacciones:   {len(df_reviews):,}")
print(f"Sparsity:        {sparsity*100:.2f}%")

fig, axes = plt.subplots(1,2,figsize=(12,4))
user_activity.hist(bins=50, color='#805AD5', edgecolor='white', ax=axes[0])
axes[0].set_title("Reviews por usuario"); axes[0].set_xlabel("Nº reseñas"); axes[0].set_yscale('log')
axes[0].axvline(user_activity.mean(), color='red', linestyle='--', label=f"Media:{user_activity.mean():.1f}")
axes[0].legend()
item_activity = df_reviews[item_col].value_counts()
item_activity.hist(bins=50, color='#DD6B20', edgecolor='white', ax=axes[1])
axes[1].set_title("Reviews por destino"); axes[1].set_xlabel("Nº reseñas"); axes[1].set_yscale('log')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig_user_item_activity.png",dpi=120,bbox_inches='tight')
plt.show()
print(f"Sparsity de la matriz: {sparsity*100:.2f}% → requiere técnicas de filtrado colaborativo")

## 5. Análisis de categorías de destinos

In [ ]:
if "Type" in df_dest.columns:
    type_vc = df_dest["Type"].value_counts()
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    axes[0].bar(type_vc.index, type_vc.values, color=plt.cm.Set3(np.linspace(0,1,len(type_vc))))
    axes[0].set_title("Tipos de Destino"); axes[0].set_xlabel("Tipo"); axes[0].set_ylabel("Cantidad")
    plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')
    axes[1].pie(type_vc.values, labels=type_vc.index, autopct='%1.1f%%',
                colors=plt.cm.Set3(np.linspace(0,1,len(type_vc))))
    axes[1].set_title("Proporción por tipo")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig_dest_types.png",dpi=120,bbox_inches='tight')
    plt.show()
else:
    print("Columna 'Type' no disponible en este dataset.")

## 6. Análisis temporal (si hay fecha)

In [ ]:
date_col = next((c for c in df_reviews.columns if 'date' in c.lower() or 'time' in c.lower() or 'visit' in c.lower()), None)
if date_col:
    df_reviews[date_col] = pd.to_datetime(df_reviews[date_col], errors='coerce')
    df_reviews_dt = df_reviews.dropna(subset=[date_col])
    monthly = df_reviews_dt.groupby(df_reviews_dt[date_col].dt.to_period('M')).size()
    plt.figure(figsize=(14,4))
    plt.plot(monthly.index.astype(str), monthly.values, color='#2C5282', linewidth=2)
    plt.title("Reseñas por mes"); plt.xlabel("Mes"); plt.ylabel("Nº reseñas")
    plt.xticks(rotation=45, ha='right'); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig_temporal.png",dpi=120,bbox_inches='tight')
    plt.show()
else:
    print("No se encontró columna de fecha en el dataset de reviews.")

## 7. Conclusiones del EDA
- Dataset real de Kaggle con reviews de destinos de viaje.
- Alta dispersión (sparsity) de la matriz → justifica NCF sobre métodos simples.
- Distribución desigual de actividad por usuario y destino → filtrado de usuarios activos.
- Rating promedio y diversidad de categorías permiten construir un sistema de recomendación personalizado.
- Los embeddings del NCF capturarán preferencias latentes no evidentes en los datos crudos.